# 02 - Factor Research

Compute the **Alpha158** feature set (qlib, from scratch) plus classical factors, then evaluate predictive power with an **alphalens-style** IC / ICIR / turnover / quantile-returns report and a factor-decay analysis.

In [1]:
import os
import sys
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
# hmmlearn reports non-convergence via logging (not warnings), so it
# bypasses the filter above; quiet it for clean notebook output.
import logging

logging.getLogger("hmmlearn").setLevel(logging.ERROR)
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Put the quantcortex repo root on the path (notebooks live in research/).
for p in ("..", "."):
    ap = os.path.abspath(p)
    if ap not in sys.path:
        sys.path.insert(0, ap)

RNG = np.random.default_rng(42)

def load_prices(symbols, start="2018-01-01", periods=1800):
    """Real prices via yfinance if available, else deterministic synthetic GBM."""
    try:
        from quantcortex.data.providers.yfinance_provider import YFinanceProvider
        px = YFinanceProvider().get_prices(list(symbols), start=start)
        if px is not None and not px.empty and px.shape[0] > 120:
            return px.dropna(how="all").ffill().dropna()
        raise RuntimeError("empty frame")
    except Exception as exc:
        print(f"[offline] yfinance unavailable ({type(exc).__name__}); using synthetic data.")
    dates = pd.bdate_range(start, periods=periods)
    mu = RNG.normal(0.0003, 0.00015, len(symbols))
    vol = RNG.uniform(0.008, 0.018, len(symbols))
    shocks = RNG.normal(mu, vol, size=(periods, len(symbols)))
    return pd.DataFrame(100 * np.exp(np.cumsum(shocks, axis=0)),
                        index=dates, columns=list(symbols))

def synth_ohlcv(close):
    """Build an OHLCV frame around a close series (synthetic intrabar range)."""
    close = close.dropna()
    hi = close * (1 + np.abs(RNG.normal(0, 0.004, len(close))))
    lo = close * (1 - np.abs(RNG.normal(0, 0.004, len(close))))
    op = close.shift(1).fillna(close.iloc[0])
    vol = RNG.integers(1_000_000, 6_000_000, len(close)).astype(float)
    return pd.DataFrame({"open": op, "high": hi, "low": lo, "close": close, "volume": vol})

print("quantcortex research environment ready.")


quantcortex research environment ready.


In [2]:
symbols = ["AAPL", "MSFT", "AMZN", "NVDA", "JPM", "XOM", "PG", "KO"]
prices = load_prices(symbols)
returns = prices.pct_change().dropna()
print("universe:", list(prices.columns), prices.shape)

universe: ['AAPL', 'MSFT', 'AMZN', 'NVDA', 'JPM', 'XOM', 'PG', 'KO'] (2124, 8)


## Alpha158 features (single symbol)

In [3]:
from quantcortex.alpha.feature_engineering.alpha158 import Alpha158

ohlcv = synth_ohlcv(prices[symbols[0]])
feats = Alpha158().compute(ohlcv)
print("Alpha158 produced", feats.shape[1], "features")
assert feats.shape[1] == 158
feats.dropna().iloc[-1].head(12)

Alpha158 produced 158 features


KMID     0.018171
KLEN     0.002315
KMID2    7.848205
KUP      0.000276
KUP2     0.119038
KLOW    -0.016131
KLOW2   -6.967243
KSFT     0.001764
KSFT2    0.761924
OPEN0    0.982154
HIGH0    1.000271
LOW0     0.997997
Name: 2026-06-15 00:00:00, dtype: float64

## Classical cross-sectional factors

In [4]:
from quantcortex.alpha.factors.classical.low_vol import LowVolFactor
from quantcortex.alpha.factors.classical.momentum import MomentumFactor

mom = MomentumFactor(lookback=126, gap=21)
mom_panel = mom.compute(prices)
mom_z = mom.cross_sectional_zscore(mom_panel)
lowvol_panel = LowVolFactor(window=63).compute(prices)
print("momentum panel:", mom_panel.shape, "| non-null rows:", int(mom_panel.notna().any(axis=1).sum()))
mom_z.dropna(how="all").tail(3)

momentum panel: (2124, 8) | non-null rows: 1998


,AAPL,MSFT,AMZN,NVDA,JPM,XOM,PG,KO
date,,,,,,,,
2026-06-11,-0.228436,-2.063625,0.558668,0.772568,-0.550422,1.465228,-0.382813,0.428833
2026-06-12,-0.154109,-1.871628,0.542383,1.025599,-0.889488,1.387722,-0.472703,0.432225
2026-06-15,-0.195860,-1.722895,0.384686,1.349069,-1.010823,1.238012,-0.505849,0.463660


## Alphalens report (IC, ICIR, turnover, quantiles)

In [5]:
from quantcortex.alpha.validation.alphalens_report import AlphalensReport

fwd_returns = prices.pct_change(5).shift(-5)   # 5-day forward return
factor = mom_z.dropna(how="all")
report = AlphalensReport(factor, fwd_returns, quantiles=4).compute()
turn = float(np.nanmean(np.asarray(report["turnover"], dtype=float)))
ls = float(np.asarray(report["long_short_return"]).mean())
print("IC mean  : %+.4f" % report["ic_mean"])
print("ICIR     : %+.4f" % report["icir"])
print("IC t-stat: %+.3f" % report["ic_tstat"])
print("turnover : %.3f" % turn)
print("long-short: %+.5f" % ls)

IC mean  : +0.0251
ICIR     : +0.0517
IC t-stat: +2.310
turnover : 0.058
long-short: +0.00237


## Factor decay

In [6]:
from quantcortex.alpha.validation.factor_decay import FactorDecay

decay = FactorDecay().compute(factor, returns, max_lag=10)
print(decay.round(4).to_string())

     ic_mean  ic_std    icir
lag                         
1     0.0222  0.4837  0.0458
2     0.0309  0.4798  0.0643
3     0.0329  0.4871  0.0676
4     0.0274  0.4874  0.0561
5     0.0251  0.4861  0.0517
6     0.0246  0.4816  0.0511
7     0.0282  0.4790  0.0588
8     0.0259  0.4799  0.0539
9     0.0278  0.4818  0.0578
10    0.0241  0.4797  0.0502


In [7]:
ic = report["ic"]
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ic.cumsum().plot(ax=ax[0]); ax[0].set_title("Cumulative IC"); ax[0].axhline(0, color="k", lw=0.5)
decay["ic_mean"].plot(kind="bar", ax=ax[1]); ax[1].set_title("Mean IC by forward lag")
plt.tight_layout(); print("factor research complete.")

factor research complete.
